> 在线讲义：[第 11 章 · 没有动作标签：隐动作与世界模型](https://wikieden.github.io/vla-from-scratch/lessons/11-latent-actions.html) · 本书《VLA 从零到 π0》

# 第 11 章 · 没有动作标签：隐动作与世界模型

配套讲义：仓库里 `vla-deep-dive/notes/lessons/11-latent-actions.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 2 步 | `encode((0, 0), (1.6, −2.0))` | z7 ↘ |
| 第 2 步 | `decode((0, 0), 7)` | (1.1, −1.1)，误差约 1.03 |
| 第 3 步 | `LAPA` | 高 6.22%、效率 30 倍以上 |

玩具设定是编的：画面 = 一个小球的位置；码本 = 8 个方向，每个码走 1.55。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
import math
ARROWS = ["→", "↗", "↑", "↖", "←", "↙", "↓", "↘"]
CODEBOOK = np.array([[math.cos(k * math.pi / 4), math.sin(k * math.pi / 4)] for k in range(8)])
STEP = 1.55
LAPA = {"比真实动作标签训练的最强 VLA 高": "6.22%", "预训练效率": "30 倍以上"}

**函数 `encode`**：编码器（逆动力学）：两帧之间的变化，选最接近的那个方向码。

In [ ]:
def encode(x_t, x_next):
    """编码器（逆动力学）：两帧之间的变化，选最接近的那个方向码。"""
    d = np.asarray(x_next, float) - np.asarray(x_t, float)
    return int(np.argmax(CODEBOOK @ (d / np.linalg.norm(d))))

**函数 `decode`**：解码器（世界模型）：这一帧 + 码 → 还原下一帧。

In [ ]:
def decode(x_t, z):
    """解码器（世界模型）：这一帧 + 码 → 还原下一帧。"""
    return np.asarray(x_t, float) + STEP * CODEBOOK[z]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 编码器 + 解码器")
x_t, x_next = (0.0, 0.0), (1.6, -2.0)
z = encode(x_t, x_next)
rec = decode(x_t, z)
err = float(np.linalg.norm(rec - np.asarray(x_next)))
print(f"  变化 (1.6, −2.0) → 码 z{z} {ARROWS[z]}，还原到 ({rec[0]:.1f}, {rec[1]:.1f})，误差 {err:.2f}")

In [ ]:
print("\n第三步 LAPA 的效果")
for k, v in LAPA.items():
    print(f"  {k}：{v}")

## 3. 代码里才看得到的

1. 编码器用的是"方向最接近"（余弦），不看距离长短；所以走远走近都会选同一个码，解码器只能还原一个固定步长 1.55。
2. 码本 8 个方向是手写的；真实 VQ-VAE 的码本是学出来的向量。

## 4. 自己改一改

- 把码本改成 16 个方向，看还原误差变小多少。
- 把 `STEP` 改成 2.56（这次变化的真实长度），看误差剩多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 视频多，动作缺：缺的那一栏自己猜。
> 两帧之间起个名，码本小，只记要紧的变化。
> 先学动作词，再学词到关节。
> 要么记代号，要么想画面。